# Build Analysis Dataset

This notebook creates the first working dataset for the project. It focuses on one TMA assessment:

- `id_assessment = 34874`
- module `FFF`
- presentation `2013J`
- deadline day `47`

The goal is one row per student submission, with score, pre-deadline engagement measures, and student controls.

In [15]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ASSESSMENT_ID = 34874

RAW_DIR

PosixPath('/Users/sunnyxie/Documents/Codex/2026-07-09/i/oulad-last-minute-learning/data/raw')

## Load Assessment And Submission Data

In [16]:
assessments = pd.read_csv(RAW_DIR / "assessments.csv")
student_assessment = pd.read_csv(RAW_DIR / "studentAssessment.csv")
student_info = pd.read_csv(RAW_DIR / "studentInfo.csv")

assessment = assessments.loc[assessments["id_assessment"] == ASSESSMENT_ID].iloc[0]
assessment

code_module            FFF
code_presentation    2013J
id_assessment        34874
assessment_type        TMA
date                    47
weight                12.5
Name: 145, dtype: object

In [17]:
deadline = int(assessment["date"])
module = assessment["code_module"]
presentation = assessment["code_presentation"]

submissions = student_assessment.loc[
    student_assessment["id_assessment"] == ASSESSMENT_ID,
    ["id_assessment", "id_student", "date_submitted", "is_banked", "score"],
].copy()

submissions["score"] = pd.to_numeric(submissions["score"], errors="coerce")
submissions = submissions.dropna(subset=["score"])

submissions.shape, submissions.head()

((1658, 5),
         id_assessment  id_student  date_submitted  is_banked  score
 117932          34874       29335              44          0   92.0
 117933          34874       29769              54          0   64.0
 117934          34874       32221              46          0   90.0
 117935          34874       34731              47          0   72.0
 117936          34874       42616              50          0   74.0)

## Build Pre-Deadline Engagement Features

For this assessment, the 28-day window is days 19 through 46. The final 7-day window is days 40 through 46.

In [18]:
window_start = deadline - 28
window_end = deadline - 1
final_7_start = deadline - 7

submitter_ids = set(submissions["id_student"])
chunks = []

usecols = ["code_module", "code_presentation", "id_student", "date", "sum_click"]

for chunk in pd.read_csv(RAW_DIR / "studentVle.csv", usecols=usecols, chunksize=500_000):
    chunk = chunk.loc[
        (chunk["code_module"] == module)
        & (chunk["code_presentation"] == presentation)
        & (chunk["id_student"].isin(submitter_ids))
        & (chunk["date"].between(window_start, window_end))
    ].copy()

    if not chunk.empty:
        chunks.append(chunk)

vle_window = pd.concat(chunks, ignore_index=True)
vle_window.shape, vle_window.head()

((178598, 5),
   code_module code_presentation  id_student  date  sum_click
 0         FFF             2013J      556639    19          1
 1         FFF             2013J      553283    19          2
 2         FFF             2013J      553364    19          3
 3         FFF             2013J      553364    19          4
 4         FFF             2013J      553283    19          1)

In [19]:
vle_window["week_window"] = pd.cut(
    vle_window["date"],
    bins=[window_start - 1, deadline - 22, deadline - 15, deadline - 8, deadline - 1],
    labels=["clicks_days_28_22", "clicks_days_21_15", "clicks_days_14_8", "clicks_days_7_1"],
)

weekly_clicks = (
    vle_window.pivot_table(
        index="id_student",
        columns="week_window",
        values="sum_click",
        aggfunc="sum",
        observed=False,
    )
    .fillna(0)
    .reset_index()
)

daily_clicks = (
    vle_window.groupby(["id_student", "date"], as_index=False)["sum_click"]
    .sum()
)

engagement = (
    vle_window.groupby("id_student", as_index=False)
    .agg(total_clicks_28d=("sum_click", "sum"))
    .merge(
        vle_window.loc[vle_window["date"] >= final_7_start]
        .groupby("id_student", as_index=False)
        .agg(final_7_clicks=("sum_click", "sum")),
        on="id_student",
        how="left",
    )
    .merge(
        daily_clicks.groupby("id_student", as_index=False)
        .agg(active_learning_days=("date", "nunique")),
        on="id_student",
        how="left",
    )
    .merge(weekly_clicks, on="id_student", how="left")
)

engagement = engagement.fillna(0)
engagement["cramming_ratio"] = np.where(
    engagement["total_clicks_28d"] > 0,
    engagement["final_7_clicks"] / engagement["total_clicks_28d"],
    0,
)

engagement.head()

,id_student,total_clicks_28d,final_7_clicks,active_learning_days,clicks_days_28_22,clicks_days_21_15,clicks_days_14_8,clicks_days_7_1,cramming_ratio
0,29335,1115,236.0,22,568,181,130,236,0.211659
1,29769,177,19.0,14,40,41,77,19,0.107345
2,32221,367,87.0,16,128,11,141,87,0.237057
3,34731,339,54.0,17,17,46,222,54,0.159292
4,42616,794,105.0,22,331,168,190,105,0.132242


## Merge Student Controls

In [20]:
controls = student_info.loc[
    (student_info["code_module"] == module)
    & (student_info["code_presentation"] == presentation),
    [
        "id_student",
        "gender",
        "region",
        "highest_education",
        "imd_band",
        "age_band",
        "num_of_prev_attempts",
        "studied_credits",
        "disability",
        "final_result",
    ],
].copy()

analysis_df = (
    submissions.merge(engagement, on="id_student", how="left")
    .merge(controls, on="id_student", how="left")
)

click_cols = [
    "total_clicks_28d",
    "final_7_clicks",
    "active_learning_days",
    "clicks_days_28_22",
    "clicks_days_21_15",
    "clicks_days_14_8",
    "clicks_days_7_1",
    "cramming_ratio",
]
analysis_df[click_cols] = analysis_df[click_cols].fillna(0)

analysis_df.shape, analysis_df.head()

((1658, 22),
    id_assessment  id_student  date_submitted  is_banked  score  \
 0          34874       29335              44          0   92.0   
 1          34874       29769              54          0   64.0   
 2          34874       32221              46          0   90.0   
 3          34874       34731              47          0   72.0   
 4          34874       42616              50          0   74.0   
 
    total_clicks_28d  final_7_clicks  active_learning_days  clicks_days_28_22  \
 0            1115.0           236.0                  22.0              568.0   
 1             177.0            19.0                  14.0               40.0   
 2             367.0            87.0                  16.0              128.0   
 3             339.0            54.0                  17.0               17.0   
 4             794.0           105.0                  22.0              331.0   
 
    clicks_days_21_15  ...  cramming_ratio  gender                region  \
 0              181

## Save The Processed Dataset

In [21]:
output_path = PROCESSED_DIR / "analysis_assessment_34874.csv"
analysis_df.to_csv(output_path, index=False)

print(f"Saved {len(analysis_df):,} rows to {output_path}")
analysis_df.describe(include="all")

Saved 1,658 rows to /Users/sunnyxie/Documents/Codex/2026-07-09/i/oulad-last-minute-learning/data/processed/analysis_assessment_34874.csv


,id_assessment,id_student,date_submitted,is_banked,score,total_clicks_28d,final_7_clicks,active_learning_days,clicks_days_28_22,clicks_days_21_15,...,cramming_ratio,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
count,1658.0,1.658000e+03,1658.000000,1658.000000,1658.000000,1658.000000,1658.000000,1658.000000,1658.000000,1658.000000,...,1658.000000,1658,1658,1658,1658,1658,1658.000000,1658.000000,1658,1658
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,2,13,5,11,3,NaN,NaN,2,4
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,M,Scotland,A Level or Equivalent,10-20,0-35,NaN,NaN,N,Pass
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1376,181,758,188,1255,NaN,NaN,1540,905
mean,34874.0,6.894738e+05,45.379976,0.025935,72.453559,473.282268,122.777443,14.912545,133.104946,110.832931,...,0.308641,NaN,NaN,NaN,NaN,NaN,0.135706,87.361279,NaN,NaN
std,0.0,5.487006e+05,8.459304,0.158989,16.971895,399.006520,110.523910,6.929262,151.009126,121.640022,...,0.219250,NaN,NaN,NaN,NaN,NaN,0.408470,35.693243,NaN,NaN
min,34874.0,2.933500e+04,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,60.000000,NaN,NaN
25%,34874.0,5.158390e+05,45.000000,0.000000,64.000000,187.000000,44.250000,10.000000,26.000000,19.000000,...,0.159110,NaN,NaN,NaN,NaN,NaN,0.000000,60.000000,NaN,NaN
50%,34874.0,5.790510e+05,46.000000,0.000000,76.000000,387.500000,96.500000,15.000000,91.000000,77.000000,...,0.256544,NaN,NaN,NaN,NaN,NaN,0.000000,60.000000,NaN,NaN
75%,34874.0,5.995828e+05,47.000000,0.000000,86.000000,668.750000,168.000000,20.000000,191.000000,163.000000,...,0.405412,NaN,NaN,NaN,NaN,NaN,0.000000,120.000000,NaN,NaN
